In [ ]:
"""Convert Thailand BMA ICARTT (.ict) ground files into AirNow-format surface
obs for MELODIES-MONET.

"""

import glob
import os
import re

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
ICT_DIR = "/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/Thailand"
ICT_GLOB = "ASIAAQ-TH-BMA-*_Ground_*.ict"
OUT_DIR = "."
TH_UTC_OFFSET = 7  # Thailand local time = UTC+7 (data times are UTC)

VAR_MAP = {
    "BP":   ("barometric_pressure", "mmHg"),   # labeled "BoilingPoint" but is
                                               # barometric pressure (mmHg).
    "CO":   ("CO", "ppmv"),
    "NO2":  ("NO2", "ppbv"),
    "O3":   ("O3", "ppbv"),
    "PM10": ("PM10", "ug/m3"),
    "PM25": ("PM2.5", "ug/m3"),
    "RH":   ("relative_humidity", "%"),
    "Temp": ("temperature", "degC"),
    "WD":   ("wind_direction", "deg"),
    "WS":   ("wind_speed", "m/s"),
}

# ICARTT missing / out-of-range flags.
MISSING = [-9999.0, -7777.0, -8888.0]
NA_STRINGS = ["-9999", "-9999.0", "-7777", "-8888", "NA", "N/A", ""]
SITE_RE = re.compile(r"^\d+BMA$")
# --------------------------------------------------------------------------

def parse_icartt(path):
    """Parse one ICARTT 1001 file.

    Returns (base_date, column_names, station_meta, data_frame), where
    station_meta maps 'NBMA' -> (lat, lon, name).
    """
    with open(path) as fh:
        lines = fh.read().splitlines()

    nlhead = int(lines[0].split(",")[0])
    y, m, d = (int(x) for x in lines[6].split(",")[:3])
    base = pd.Timestamp(year=y, month=m, day=d)

    header = lines[:nlhead]
    colnames = [c.strip() for c in header[-1].split(",")]

    # Station metadata: the block starting at the "stationID,..." line.
    meta = {}
    for i, ln in enumerate(header):
        if ln.lower().startswith("stationid,"):
            for row in header[i + 1:]:
                parts = [p.strip() for p in row.split(",")]
                if len(parts) >= 3 and SITE_RE.match(parts[0]):
                    try:
                        meta[parts[0]] = (
                            float(parts[1]), float(parts[2]),
                            parts[3] if len(parts) > 3 else "",
                        )
                    except ValueError:
                        break
                else:
                    break  # end of the station table
            break

    data = pd.read_csv(path, skiprows=nlhead, names=colnames, na_values=NA_STRINGS)
    return base, colnames, meta, data


def load_file(path, global_meta):
    base, cols, _meta, data = parse_icartt(path)
    time_utc = base + pd.to_timedelta(
        pd.to_numeric(data["Time_Start"], errors="coerce"), unit="s"
    )
    value_cols = [c for c in cols if c not in ("Time_Start", "Time_Stop")]
    rows = []
    for c in value_cols:
        if "_" not in c:
            continue
        prefix, siteid = c.split("_", 1)
        std, units = VAR_MAP.get(prefix, (prefix, "unknown"))
        lat, lon, name = global_meta.get(siteid, (np.nan, np.nan, ""))
        vals = pd.to_numeric(data[c], errors="coerce").replace(MISSING, np.nan)
        if vals.notna().sum() == 0:
            continue
        rows.append(
            pd.DataFrame(
                {
                    "time": time_utc.values,
                    "siteid": siteid,
                    "site": name,
                    "latitude": lat,
                    "longitude": lon,
                    "variable": std,
                    "units": units,
                    "obs": vals.values,
                    "network": "TH_BMA",
                }
            )
        )
    if not rows:
        return None
    out = pd.concat(rows, ignore_index=True)
    return out.dropna(subset=["time", "obs"])


def to_airnow_long(df):
    df = df.copy()
    df["utcoffset"] = TH_UTC_OFFSET
    df["time"] = pd.to_datetime(df["time"])
    df["time_local"] = df["time"] + pd.to_timedelta(TH_UTC_OFFSET, unit="h")
    for col in ["cmsa_name", "msa_code", "msa_name", "state_name", "epa_region"]:
        df[col] = np.nan
    df["siteid"] = df["siteid"].astype(str)
    cols = [
        "time", "siteid", "site", "utcoffset", "variable", "units", "obs",
        "time_local", "latitude", "longitude", "cmsa_name", "msa_code",
        "msa_name", "state_name", "epa_region", "network",
    ]
    return df[cols].sort_values(["siteid", "variable", "time"]).reset_index(drop=True)


def long_to_wide(df_long):

    idx = ["time", "siteid", "network", "utcoffset", "time_local"]
    wide = (
        df_long.pivot_table(index=idx, columns="variable", values="obs", aggfunc="mean")
        .reset_index()
    )
    wide.columns.name = None
    site_meta = (
        df_long.groupby("siteid")[["latitude", "longitude", "site"]]
        .first()
        .reset_index()
    )
    return wide.merge(site_meta, on="siteid", how="left")


def wide_to_netcdf(df_wide, path):

    meta_cols = ["site", "latitude", "longitude", "utcoffset", "network", "time_local"]
    var_cols = [c for c in df_wide.columns
                if c not in (["time", "siteid"] + meta_cols)]

    df = df_wide.copy()
    sites = df["siteid"].drop_duplicates().tolist()
    df["x"] = df["siteid"].map({s: i for i, s in enumerate(sites)})

    ds = df.set_index(["x", "time"])[var_cols].to_xarray()  # dims (x, time)

    site_meta = df.drop_duplicates("x").set_index("x").sort_index().reindex(ds["x"].values)
    ds = ds.assign_coords(
        # cast to plain str (not pandas StringDtype) so NetCDF encoding works
        siteid=("x", site_meta["siteid"].astype(str).to_numpy(dtype=object)),
        latitude=("x", site_meta["latitude"].to_numpy(dtype="float64")),
        longitude=("x", site_meta["longitude"].to_numpy(dtype="float64")),
        utcoffset=("x", site_meta["utcoffset"].to_numpy(dtype="float64")),
    )
    ds.to_netcdf(path)
    return ds


In [ ]:
files = sorted(glob.glob(os.path.join(ICT_DIR, ICT_GLOB)))
if not files:
    raise SystemExit(f"No .ict files under {ICT_DIR}/{ICT_GLOB}")
print(f"Found {len(files)} ICARTT files")

global_meta = {}
for f in files:
    _, _, meta, _ = parse_icartt(f)
    for sid, ll in meta.items():
        global_meta.setdefault(sid, ll)
print(f"Stations with metadata: {len(global_meta)}")

all_long = []
for f in files:
    r = load_file(f, global_meta)
    if r is not None:
        print(f"  {os.path.basename(f)}: {len(r):,} rows, "
              f"vars={sorted(r['variable'].unique())}")
        all_long.append(r)
if not all_long:
    raise SystemExit("No data read.")

long = to_airnow_long(pd.concat(all_long, ignore_index=True))
wide = long_to_wide(long)

n_nocoord = long.loc[long["latitude"].isna(), "siteid"].nunique()
print(f"\nTotal long rows: {len(long):,}")
print("Variables:", sorted(long["variable"].unique()))
print("Sites:", long["siteid"].nunique(),
